# Filter the sequence-matched deposition pairs

Run after `offset_vs_inherent_1_decomposition.ipynb`, which writes `decomposition.json`. Removes pairs that differ in binding or other sample state, and writes `decomposition_auto_only.json` (automatic filters) and `decomposition_no_othernoise.json` (automatic filters and manual review). Next: `offset_vs_inherent_2_distributions.ipynb`, `hsqc_offset_plots.ipynb`.

In [ ]:
import json
import re
import glob
import os
import csv

HERE = os.getcwd()
DECOMP = os.path.join(HERE, "../../data/analysis/Fig3/inherent_test/decomposition.json")
BMRB_DIR = "../../data/rBMRB/bmrb_entries"

# keyword sets applied to the lower-cased assembly title
DENAT_TERMS = ["denatured", "denaturated", "unfolded", "urea-denatured", "acid-denatured",
                "thermally denatured", "chemically denatured", "guanidine", "gdnhcl", "gdmcl",
                "random coil", "molten globule", "intrinsically disordered"]
NATIVE_TERMS = ["native", "folded", "native-state"]
UNFOLDED_VALUES = {"unfolded", "partially unfolded", "intrinsically disordered",
                    "molten globule", "partially disordered", "folding intermediate"}

# Keywords are matched as whole words with regex word boundaries instead of plain substring `in`
# (avoids false hits such as "bound" in "unbound", "rna" in "alternative", "dna" in "DnaJ")
BOUND_TERMS = [r"bound", r"complex", r"complexed", r"holo", r"gdp", r"gtp",
               r"gmppnp", r"gppnhp", r"gtpgammas", r"amppnp", r"atp", r"adp",
               r"substrate", r"ligand", r"cofactor", r"coenzyme",
               r"inhibitor", r"activator", r"dna", r"rna"]
FREE_TERMS = [r"free", r"apo", r"unbound", r"isolated", r"alone",
              r"nucleotide-free", r"uncomplexed", r"unliganded"]
OLIGO_TERMS = [r"dimer", r"trimer", r"tetramer", r"pentamer", r"hexamer",
               r"heptamer", r"octamer", r"multimer", r"oligomer"]

def _word_hit(term, low):
    """term must match as a separate word: preceded by a word boundary (excludes 'bound' in 'unbound',
    'rna' in 'alternative', 'dna' in 'DnaJ'); suffixes such as s/es/ed/ic are allowed."""
    return re.search(rf"(?<![a-zA-Z]){re.escape(term)}(?![a-zA-Z])", low) is not None

import shlex

_EA_LOOP = re.compile(r"loop_\s*\n((?:\s*_Entity_assembly\.\S+\s*\n)+)(.*?)\n\s*stop_", re.S)

OX_TERMS = [r"oxidi[sz]ed", r"oxidi[sz]ation", r"ox\b", r"disulfide[- ]bonded", r"disulphide[- ]bonded"]
RED_TERMS = [r"reduced", r"reduction", r"red\b", r"dithiol", r"free[- ]thiol"]

THIOL_UNKNOWN = {"", ".", "?", "na", "n/a", "unknown", "not available",
                "not reported", "not present"}

_THIOL_TAGS = ("_Assembly.Thiol_state", "_Entity.Thiol_state")


def _thiol_state(text):
    """First informative Thiol_state, e.g. 'all free' / 'all disulfide bound' ('' if absent).

    _Assembly.Thiol_state is '.' in most entries, so fall through to _Entity.Thiol_state;
    an uninformative value ('.', 'not present', ...) is still returned when nothing better
    exists, so the raw field stays visible -- classify_pair filters it via THIOL_UNKNOWN.
    """
    first = ""
    for tag in _THIOL_TAGS:
        for m in re.finditer(rf"{re.escape(tag)}\s+('([^']*)'|\"([^\"]*)\"|(\S+))", text):
            v = " ".join((m.group(2) or m.group(3) or m.group(4) or "").split()).lower()
            first = first or v
            if v not in THIOL_UNKNOWN:
                return v
    return first

def redox_of(text):
    low = re.sub(r"\bredox\b", " ", text.lower())          # "redox protein" is a neutral description, not a state
    ox = any(re.search(rf"(?<![a-z]){p}", low) for p in OX_TERMS)
    red = any(re.search(rf"(?<![a-z]){p}", low) for p in RED_TERMS)
    return "ox" if (ox and not red) else "red" if (red and not ox) else None

def _sources(info):
    """Sources are ordered from most to least specific; stop at the first one that determines the state."""
    return [info["name"], " ".join(info["cs_list_names"]), info["title"]]

def redox_pair(info_a, info_b):
    """Compare both entries on sources of the same priority; return (ra, rb, src) or None."""
    for src, (ta, tb) in zip(("assembly name", "shift-list name", "entry title"),
                             zip(_sources(info_a), _sources(info_b))):
        ra, rb = redox_of(ta), redox_of(tb)
        if ra and rb and ra != rb:
            return ra, rb, src
    return None

def _physical_states(text):
    """Values of _Entity_assembly.Physical_state (one per component), e.g. ['denatured']."""
    states = []
    for m in _EA_LOOP.finditer(text):
        tags = m.group(1).split()
        if "_Entity_assembly.Physical_state" not in tags:
            continue
        i = tags.index("_Entity_assembly.Physical_state")
        for line in m.group(2).splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            try:
                fields = shlex.split(line)
            except ValueError:
                continue
            if len(fields) == len(tags):
                states.append(fields[i].lower())
    return states

def read_entry(eid):
    """eid like 'bmr17477_1' -> assembly title + entry title + component count from the .str file."""
    match = re.match(r"(bmr\d+)_\d+", eid)
    base = match.group(1) if match else eid
    files = sorted(glob.glob(f"{BMRB_DIR}/{base}/{base}_*.str"))
    if not files:
        return dict(name="", title="", n_components=0, found=False)
    text = open(files[0], errors="ignore").read()

    name_match = re.search(r"_Assembly\.Name\s+('([^']*)'|\"([^\"]*)\"|(\S+))", text)
    name = ""
    if name_match:
        name = (name_match.group(2) or name_match.group(3) or name_match.group(4) or "").strip()

    # _Entry.Title is a semicolon-delimited multi-line block; fall back to the quoted form
    title = ""
    t_block = re.search(r"_Entry\.Title\s*\n;\s*\n(.*?)\n;", text, re.S)
    if t_block:
        title = " ".join(t_block.group(1).split())
    else:
        t_inline = re.search(r"_Entry\.Title\s+('([^']*)'|\"([^\"]*)\"|(\S+))", text)
        if t_inline:
            title = " ".join((t_inline.group(2) or t_inline.group(3) or t_inline.group(4) or "").split())

    # number of rows in the _Entity_assembly loop = number of named components in the tube
    n_components = len(re.findall(r"_Entity_assembly\.Entity_assembly_name", text))
    states = _physical_states(text)
    thiol = _thiol_state(text)

    # _Assigned_chem_shift_list.Name often carries the state when the title does not
    cs_names = re.findall(r"_Assigned_chem_shift_list\.Name\s+('([^']*)'|\"([^\"]*)\"|(\S+))", text)
    cs_list_names = [" ".join((g[1] or g[2] or g[3] or "").split()) for g in cs_names]

    return dict(name=name, title=title, cs_list_names=cs_list_names,
                n_components=n_components, phys_states=states,
                thiol_state=thiol,
                denatured=any(s in UNFOLDED_VALUES for s in states),
                phys_known=bool(states) and all(s not in (".", "?") for s in states),
                found=True)

def state_of(name):
    """Coarse binding state from an assembly title."""
    low = name.lower()
    is_bound = any(_word_hit(term, low) for term in BOUND_TERMS)
    # "protein/ligand" notation (e.g. lc-ltp2/DMPG) = bound; ignore pure domain ranges like 2-3
    if re.search(r"[a-z0-9]/[a-z]", low):
        is_bound = True
    is_free = any(_word_hit(term, low) for term in FREE_TERMS)
    oligo_hits = [term for term in OLIGO_TERMS
              if re.search(rf"{re.escape(term)}(ic|s)?\b", low)]
    # With word-boundary matching, "unbound"/"uncomplexed"/"unliganded" no longer trigger is_bound;
    # keep a safety net here using the same boundary test rather than substring containment
    if is_free:
        is_bound = is_bound and (_word_hit("bound", low) or _word_hit("complex", low))

    denat_hits = [term for term in DENAT_TERMS if term in low]
    native_hits = [term for term in NATIVE_TERMS if re.search(rf"\b{re.escape(term)}\b", low)]

    return dict(bound=is_bound, free=is_free, oligo=oligo_hits,
                denat=bool(denat_hits), denat_terms=denat_hits, native=bool(native_hits))

def classify_pair(info_a, info_b):
    """Return (call, reason). call in {binding, oligomer, other/noise, unknown}."""
    if not (info_a["found"] and info_b["found"]):
        return "unknown", "missing star file"
    def _text(info):
        return " || ".join([info["name"], info["title"]] + info["cs_list_names"])

    sa = state_of(_text(info_a))
    sb = state_of(_text(info_b))

    if sa["denat"] != sb["denat"] or sa["native"] != sb["native"]:
        return "denatured", "title language: denatured/native differs"

    # folding state from the structured _Entity_assembly.Physical_state field
    if info_a["phys_known"] and info_b["phys_known"] and info_a["denatured"] != info_b["denatured"]:
        d, n = ((info_a, info_b) if info_a["denatured"] else (info_b, info_a))
        return "denatured", f"Physical_state: {'/'.join(d['phys_states'])} vs {'/'.join(n['phys_states'])}"

    # redox state from the structured _Assembly.Thiol_state field
    ta, tb = info_a["thiol_state"], info_b["thiol_state"]
    if ta not in THIOL_UNKNOWN and tb not in THIOL_UNKNOWN and ta != tb:
        return "redox", f"Thiol_state: {ta} vs {tb}"

    hit = redox_pair(info_a, info_b)
    if hit:
        ra, rb, src = hit
        return "redox", f"redox state: {ra} vs {rb} (from {src})"

    # explicit bound-vs-free (ligand / partner binding)
    bound_free_split = (sa["bound"] and sb["free"]) or (sb["bound"] and sa["free"])
    one_bound = sa["bound"] != sb["bound"]                       # one titled as complex, other not
    comp_diff = (info_a["n_components"] > 1) != (info_b["n_components"] > 1)
    if bound_free_split:
        return "binding", "bound vs free in title"
    if one_bound or comp_diff:
        return "binding", "one is a complex/bound, other not"

    # oligomerisation-state difference (self-association)
    if sa["oligo"] != sb["oligo"]:
        return "oligomer", "oligomeric-state difference"

    return "other/noise", "no binding language differs"


records = json.load(open(DECOMP))
print(f"loaded {len(records)} records from {DECOMP}")
best = {}
for r in records:
    key = tuple(sorted([r["eid_a"], r["eid_b"]]))
    if key not in best or r["corr_rmse"] > best[key]["corr_rmse"]:
        best[key] = r
pairs = sorted(best.values(), key=lambda x: -x["corr_rmse"])

rows = []
for r in pairs:
    info_a = read_entry(r["eid_a"])
    info_b = read_entry(r["eid_b"])
    call, reason = classify_pair(info_a, info_b)
    rows.append(dict(corr_rmse=r["corr_rmse"], eid_a=r["eid_a"], eid_b=r["eid_b"],
                        n_common=r["n_common"], name_a=info_a["name"], name_b=info_b["name"],
                        title_a=info_a["title"], title_b=info_b["title"],
                        call=call, reason=reason))

# ---- report ----
print(f"{len(rows)} unique pairs (sorted by corr_rmse desc)\n")
print(f"{'#':>3} {'corr':>5} {'pair':<26} {'call':<12} names")
for i, row in enumerate(rows, 1):
    tag = "  <<<" if i <= 5 else ""
    # if row["call"] != "other/noise":
    #     continue
    print(f"{i:>3} {row['corr_rmse']:5.2f} {row['eid_a']+' / '+row['eid_b']:<26} "
            f"{row['call']:<12} {row['name_a'][:60]!r} | {row['name_b'][:60]!r}{tag} ({row['reason']})")

def summarize(subset, label):
    n = len(subset)
    binding = sum(1 for r in subset if r["call"] == "binding")
    oligo = sum(1 for r in subset if r["call"] == "oligomer")
    other = sum(1 for r in subset if r["call"] == "other/noise")
    unknown = sum(1 for r in subset if r["call"] == "unknown")
    denat = sum(1 for r in subset if r["call"] == "denatured")
    redox = sum(1 for r in subset if r["call"] == "redox")
    print(f"\n{label} (n={n}): denatured={denat}  binding={binding} ({binding/n:.0%})  "
            f"oligomer={oligo}  other/noise={other}  redox={redox}  unknown={unknown}")


print("\n" + "=" * 60)
summarize(rows[:5], "TOP 5")
summarize(rows[:20], "TOP 20")
summarize(rows, "ALL PAIRS")

In [ ]:
OUT_JSON = os.path.join(HERE, "../../data/analysis/Fig3/inherent_test/decomposition_no_othernoise.json")

# Classify each unordered pair once (cached), then decide every record
call_of = {}
for r in records:
    key = tuple(sorted([r["eid_a"], r["eid_b"]]))
    if key not in call_of:
        call_of[key] = classify_pair(read_entry(key[0]), read_entry(key[1]))[0]

kept, dropped = [], []
# "3. solid state, 4, 5, 6,7 . redox, 8. bmr18412_1 / bmr30614_1 binding, 9. bmr15264_1 / bmr16265_1 redox, 
# 10. bmr17477_1 / bmr15558_1 phosphate, 11. bmr25879_1 / bmr25875_1 binding, 12.  bmr17892_1 / bmr6317_1 dimer both "
# 13. bmr6392_1 / bmr25663_1 binding, 14. bmr50583_1 / bmr50582_1 bind, 15. bmr51710_1 / bmr50835_1  phosphalytion
# 16. bmr15895_1 / bmr15894_1  redox, 17. bmr12038_1 / bmr12035_1 phosphalated 18. bmr27238_1 / bmr27237_1 S-nitrosylated
# 18. bmr30404_1 / bmr27396_1  myo-inositol hexakisphosphate 19. bmr50484_1 / bmr50485_1: S-nitrosated
# 20. bmr50896_1 / bmr50895_1 modification 21. bmr18631_1 / bmr18630_1 redox 22 bmr51018_1 / bmr51019_1  phospholated
# 23. bmr12039_1 / bmr12036_1 phosphalated 24. bmr34383_1 / bmr25272_1 bind, 25. bmr51315_1 / bmr51314_1 methylated
# 26. bmr16805_1 / bmr18263_1 acyl carrier
#
# Third review: the two pairs below were confirmed to have no real state difference and were moved from MANUAL_DROP to KEPT:
#   - bmr17892_1/bmr6317_1 (CylR2): same homodimer at the same pH/temperature/pressure; the latter is only the
#     298 K reference point of a cold-denaturation temperature series, consistent with other 'no real difference'
#     cases such as bmr16805_1/bmr18263_1, bmr18412_1/bmr30614_1 and bmr30419_1/bmr30420_1
#   - bmr6321_1/bmr6322_1 (delta25-PYP): added by mistake in the second review; kept

# Pairs manually judged to differ in state (order-independent: stored as sorted tuples)
MANUAL_DROP = {tuple(sorted(p)) for p in [
    # --- original list ---
    ('bmr6709_1',  'bmr15280_1'),   # known binding pair, but the title does not say so
    ('bmr6625_1',  'bmr15599_1'),   # Pi bound and free form
    ('bmr51919_1', 'bmr51938_1'),   # solid state
    ('bmr6847_1',  'bmr6603_1'),    # redox
    ('bmr50861_1', 'bmr50862_1'),   # redox
    ('bmr34344_1', 'bmr50362_1'),   # redox
    ('bmr6075_1',  'bmr6563_1'),    # redox
    ('bmr15264_1', 'bmr16265_1'),   # redox
    ('bmr25879_1', 'bmr25875_1'),   # binding
    ('bmr6392_1',  'bmr25663_1'),   # binding
    ('bmr50583_1', 'bmr50582_1'),   # binding
    ('bmr51710_1', 'bmr50835_1'),   # phosphorylation
    ('bmr12038_1', 'bmr12035_1'),   # phosphorylated
    ('bmr27238_1', 'bmr27237_1'),   # S-nitrosylated
    ('bmr30404_1', 'bmr27396_1'),   # myo-inositol hexakisphosphate
    ('bmr50484_1', 'bmr50485_1'),   # S-nitrosated
    ('bmr50896_1', 'bmr50895_1'),   # modification
    ('bmr18631_1', 'bmr18630_1'),   # redox
    ('bmr51018_1', 'bmr51019_1'),   # phosphorylated
    ('bmr12039_1', 'bmr12036_1'),   # phosphorylated
    ('bmr51315_1', 'bmr51314_1'),   # methylated
    # --- added ---
    ('bmr11547_1', 'bmr11505_1'),   # Q41N mutant at 1 atm vs wt at 2500 bar (both N2 alternative state; no enzyme in either sample)
    ('bmr27269_1', 'bmr30312_1'),   # phosphate/TCEP buffer
    ('bmr18012_1', 'bmr25448_1'),   # urea
    ('bmr26573_1', 'bmr25632_1'),   # binding partner GAT
    ('bmr50864_1', 'bmr50863_1'),   # TCEP usage
    ('bmr28009_1', 'bmr28008_1'),   # different sample?
    ('bmr26574_1', 'bmr25602_1'),   # binding partner
    ('bmr19189_1', 'bmr25333_1'),   # different lipids
    ('bmr18680_1', 'bmr18679_1'),   # dT7 difference
    ('bmr19481_1', 'bmr19480_1'),   # phosphorylation
    ('bmr15975_1', 'bmr15930_1'),   # buffer concentration
    ('bmr26554_1', 'bmr16445_1'),   # SDS micelle
    ('bmr5465_1',  'bmr6158_1'),    # detergent micelle
    ('bmr16166_1', 'bmr16800_1'),   # TFE difference
    ('bmr11050_1', 'bmr11049_1'),   # DPC/SDS micelle
    ('bmr16799_1', 'bmr16798_1'),   # TFE difference
    ('bmr25817_1', 'bmr25818_1'),   # SDS/DPC micelle
    ('bmr18226_1', 'bmr18224_1'),   # SDS/LPPG micelle
    ('bmr4209_1',  'bmr4197_1'),    # different micelles

    # --- added after the second review (buffer / PTM / linkage / mutation, ...) ---
    ('bmr16275_1', 'bmr20082_1'),   # micelle: DPC vs SDS
    ('bmr15975_1', 'bmr15936_1'),   # TFE concentration series
    ('bmr52545_1', 'bmr52546_1'),   # DPC micelle presence
    ('bmr31111_1', 'bmr30979_1'),   # disulfide isomer (B also lists component Def13_B2, possibly a complex)
    ('bmr19743_1', 'bmr19742_1'),   # L-citrulline-bound vs apo DM-PaDDAH (R40E/R98H; C249 in both sequences): 19742 = identical copy of 19744 (assembly incl. L-citrulline), 19743 = DM apo (~BMRB 6074); Rasheed 2014 Biochemistry
    ('bmr15936_1', 'bmr15930_1'),   # TFE concentration series
    ('bmr30662_1', 'bmr30664_1'),   # I24N point mutant vs WT
    ('bmr30234_1', 'bmr52115_2'),   # K27 vs K33-linked diubiquitin
    ('bmr51270_1', 'bmr51271_1'),   # myristoylated vs non-myristoylated
    ('bmr30234_1', 'bmr52108_2'),   # K27 vs K29-linked diubiquitin
    ('bmr51269_1', 'bmr51280_1'),   # myristoylated vs non-myristoylated
    ('bmr30234_1', 'bmr52139_1'),   # K27 vs K48-linked diubiquitin
    ('bmr52078_1', 'bmr30234_1'),   # K6 vs K27-linked diubiquitin
    ('bmr51305_1', 'bmr51306_1'),   # redox: reduced vs neutral (component list confirms DTT)
    ('bmr30435_1', 'bmr27365_1'),   # 20% TFE vs 250 mM DPC micelle (different co-solvent)
    ('bmr51277_1', 'bmr51276_1'),   # myristoylated vs non-myristoylated
    ('bmr51274_1', 'bmr51275_1'),   # myristoylated vs non-myristoylated
    ('bmr52078_1', 'bmr52115_2'),   # K6 vs K33-linked diubiquitin
    ('bmr52078_1', 'bmr52108_2'),   # K6 vs K29-linked diubiquitin
    ('bmr26746_1', 'bmr30077_1'),   # GMPPNP-bound vs apo (component list confirms GMPPNP)
    ('bmr52139_1', 'bmr52115_2'),   # K48 vs K33-linked diubiquitin
    ('bmr52108_2', 'bmr52139_1'),   # K29 vs K48-linked diubiquitin
    ('bmr51273_1', 'bmr51272_1'),   # myristoylated vs non-myristoylated
    ('bmr51279_1', 'bmr51278_1'),   # myristoylated vs non-myristoylated
    ('bmr11519_1', 'bmr11521_1'),   # crowding (protein G B1 as crowder)
    ('bmr1813_1',  'bmr1812_1'),    # redox state (reduced/oxidized thioredoxin)
    ('bmr50679_1', 'bmr50676_1'),   # PPACK covalent active-site inhibitor
    ('bmr26894_1', 'bmr26893_1'),   # crowding; component list confirms added lysozyme
    ('bmr18037_1', 'bmr17701_1'),   # imipenem covalent acyl-enzyme
    ('bmr27709_1', 'bmr27543_1'),   # different micelles (TDPC:SDS vs LMPG)
    ('bmr52353_1', 'bmr52351_1'),   # different acyl chain length (C10 vs C6)
    ('bmr52352_1', 'bmr52351_1'),   # different acyl chain length (C8 vs C6)
    ('bmr27915_1', 'bmr27916_1'),   # homodimer vs heterodimer (extra p65 component)
    ('bmr15176_1', 'bmr15865_1'),   # different point mutations (D5G,C72S vs C72S)
    ('bmr26967_1', 'bmr26968_1'),   # glycosylation (GlcNAc) difference
    ('bmr18567_1', 'bmr18969_1'),   # different mutation sites (S114A vs S55A)
    ('bmr52078_1', 'bmr52139_1'),   # K6 vs K48-linked diubiquitin
    ('bmr27722_2', 'bmr27721_2'),   # bound to different ligands (compound R vs S)
    ('bmr16603_1', 'bmr7432_1'),    # same mutant (D39A); HscB binding not confirmed in the sample, reason uncertain
    ('bmr36110_1', 'bmr36109_1'),   # PolySia ligand present vs absent (both in 40% TFE)
    ('bmr11035_1', 'bmr11037_1'),   # in-cell (M9 slurry, confirmed) vs in vitro
    ('bmr52353_1', 'bmr52352_1'),   # different acyl chain length (C10 vs C8)
    ('bmr50199_1', 'bmr50198_1'),   # different cell lines (in-cell environment)
    ('bmr51307_1', 'bmr51306_1'),   # redox: oxidized vs neutral (component list confirms H2O2)
    ('bmr25910_1', 'bmr25909_1'),   # in-cell vs in vitro
    ('bmr52108_2', 'bmr52115_2'),   # K29 vs K33-linked diubiquitin
    ('bmr11520_1', 'bmr11521_1'),   # different crowding agents
    # --- third review: environment/covalent differences among previously kept pairs ---
    ('bmr19947_1', 'bmr19963_1'),   # bitistatin A vs B: different disulfide connectivity (5-24/16-34/18-29 vs 5-34/16-29/18-24)
    ('bmr51063_1', 'bmr51064_1'),   # NaKd18 in 100 mM K+ vs 600 mM Na+ (ion-dependent filter structure)
    ('bmr26672_1', 'bmr27887_1'),   # FUS LC dispersed vs LLPS condensed phase (477 mg/mL)
    ('bmr11519_1', 'bmr11520_1'),   # villin headpiece dilute vs crowded (32 mM self-crowding)
    ('bmr25179_1', 'bmr25586_1'),   # wt NS4A in 100 mM SDS micelle vs aqueous solution
    ('bmr25180_1', 'bmr25676_1'),   # mut NS4A (L6E;M10E) in 100 mM SDS micelle vs aqueous solution
    ('bmr17724_1', 'bmr18740_1'),   # KSR1 CA1-CA1a in SDS (82 mM) vs LMPG (300 mM) micelle
    ('bmr30416_1', 'bmr30435_1'),   # CSP1 in 250 mM DPC micelle vs 20% TFE (same contrast as bmr27365_1/bmr30435_1)
    # --- fourth review: source paper check ---
    ('bmr18412_1', 'bmr30614_1'),   # PEA-15 free vs ERK2-bound (30614 = 1:1 complex with unlabeled ERK2, Int J Mol Sci 2019 20:3335 Methods 4.3-4.4); C-terminal tail binds ERK2 DRS
    ('bmr19615_1', 'bmr19616_1'),   # TM-PaDDAH (R40E/R98H/C249S) apo (19615, ~DM apo 19743) vs L-citrulline-bound (19616, ~DM holo 19744); lid G17-D27 shifts; Rasheed 2014 Biochemistry
]}

# Pairs forced to be kept (overrides MANUAL_DROP and the automatic call)
MANUAL_KEEP = {tuple(sorted(p)) for p in [
]}

NEAR_DUP_RMSE = 0.01   # ppm, same as the original DUP_RMSE setting

# ---- first pass: automatic filters only (near-duplicate + classifier), no MANUAL_KEEP / MANUAL_DROP ----
AUTO_JSON = os.path.join(HERE, "../../data/analysis/Fig3/inherent_test/decomposition_auto_only.json")
pair_key = lambda r: tuple(sorted([r["eid_a"], r["eid_b"]]))
auto_kept = [r for r in records
             if r["corr_rmse"] >= NEAR_DUP_RMSE and call_of[pair_key(r)] == "other/noise"]
with open(AUTO_JSON, "w") as fh:
    json.dump(auto_kept, fh, indent=2)
print(f"auto pass: {len(records)} -> kept {len(auto_kept)} records "
      f"({len({pair_key(r) for r in auto_kept})} unique pairs); saved {AUTO_JSON}")


for r in records:
    key = tuple(sorted([r["eid_a"], r["eid_b"]]))
    if key in MANUAL_KEEP:
        kept.append(r)
    elif key in MANUAL_DROP:
        dropped.append(r)
    elif r["corr_rmse"] < NEAR_DUP_RMSE:
        dropped.append(r)
    else:
        (dropped if call_of[key] != "other/noise" else kept).append(r)

with open(OUT_JSON, "w") as fh:
    json.dump(kept, fh, indent=2)

n_pairs_dropped = len({tuple(sorted([r["eid_a"], r["eid_b"]])) for r in dropped})
print(f"\nrecords: {len(records)} -> kept {len(kept)}, dropped {len(dropped)} "
      f"({n_pairs_dropped} unique pairs called other/noise)")

from collections import Counter
print("dropped records by method:", dict(Counter(r.get("method", "?") for r in dropped)))
print("kept    records by method:", dict(Counter(r.get("method", "?") for r in kept)))
print(f"saved {OUT_JSON}")

In [ ]:
import json
with open("../../data/analysis/Fig3/inherent_test/decomposition_no_othernoise.json", "r") as fh:
    decomp = json.load(fh)

print("records:", len(decomp))
print("unique pairs:", len({frozenset((r["eid_a"], r["eid_b"])) for r in decomp}))

corr_rmse = {}
for r in decomp:
    corr_rmse[r["eid_a"] + "_" + r["eid_b"]] = r["corr_rmse"]

corr_rmse_sorted = sorted(corr_rmse.items(), key=lambda x: -x[1])
for i, (pair, corr) in enumerate(corr_rmse_sorted[:], 1):
    print(f"{i:>2}: {pair} -> corr_rmse={corr:.3f}")
